In [1]:
import clickhouse_connect
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from matplotlib.ticker import FuncFormatter
import datetime
import geopandas as gpd

In [2]:
# Connect to ClickHouse with resource limits
client = clickhouse_connect.get_client(
    host='192.168.1.95',
    port=8123,
    username='default',
    password='',
    database='ceir_gold',
    settings={
        'max_execution_time': 120,
        'max_memory_usage': 2000000000,  # 2GB max
        'max_threads': 2,
        'priority': 5
    }
)

# Step 1: Fetch data from the gsma_devices table
gsma_query = """
SELECT
    tac,
    device_name,
    model_name,
    device_type,
    operating_system,
    manufacturer
FROM gsma_devices
"""
gsma_df = client.query_df(gsma_query)


# Step 2: Fetch data from the domestic_subscribers table
domestic_query = """
SELECT
    msisdn,
    imsi,
    imei,
    device_type
FROM domestic_subscribers
WHERE last_seen >= now() - INTERVAL 90 DAY
ORDER BY last_seen DESC
LIMIT 10000
"""
domestic_df = client.query_df(domestic_query)

# Step 3: Fetch data from the roamers table
roam_query = """
SELECT
    msisdn,
    imsi,
    imei,
    device_type
FROM roamers
WHERE last_seen >= now() - INTERVAL 90 DAY
ORDER BY last_seen DESC
LIMIT 10000
"""
roam_df = client.query_df(roam_query)
